# Validate the LLM judge on Kaggle (clinical-rag-eval, issue #9)

Serves **Qwen/Qwen3-8B-AWQ** with vLLM on one Kaggle T4 (16 GB) and asks it to label PubMedQA's 1,000 reference conclusions (`long_answer`) as yes / no / maybe. Agreement with the experts' `final_decision` (accuracy, Cohen's kappa, confusion matrix) is written to `/kaggle/working/results/`.

**Settings:** Accelerator = GPU T4 x2 (one GPU is used), Internet = on. Run all cells (about 20-40 minutes), then download `results/m3_judge_validate_qwen3-8b-awq.json` and `.jsonl` from the Output tab and add them to the repo's `results/` folder.

In [ ]:
%%bash
pip install -q vllm
git clone -q https://github.com/Arjun-Sivasankar/clinical-rag-eval.git
pip install -q -e clinical-rag-eval
nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
# Start the OpenAI-compatible server in the background. T4 has no bf16, so fp16 (--dtype half).
cd /kaggle/working
nohup python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-8B-AWQ --dtype half \
  --max-model-len 2048 --gpu-memory-utilization 0.90 --port 8000 > vllm.log 2>&1 &
for i in $(seq 1 120); do
  curl -sf localhost:8000/v1/models >/dev/null && echo "server ready" && break
  sleep 10
done
tail -3 vllm.log

In [ ]:
%%bash
cd /kaggle/working/clinical-rag-eval
export CRAG_RESULTS_DIR=/kaggle/working/results CRAG_DATA_DIR=/kaggle/working/data
python -m clinical_rag.cli prepare
python -m clinical_rag.cli judge-validate --base-url http://localhost:8000/v1 \
  --model Qwen/Qwen3-8B-AWQ --backend "vllm $(python -c 'import vllm; print(vllm.__version__)')" \
  --judge-hardware "Kaggle Tesla T4 16 GB" --name m3_judge_validate_qwen3-8b-awq

In [ ]:
import json
r = json.load(open("/kaggle/working/results/m3_judge_validate_qwen3-8b-awq.json"))
print(json.dumps(r["agreement"], indent=2))